# Lab 8 — Ship It

**Module:** Lightweight Deployment & Final Evaluation | **Duration:** 90 min

**Fil Rouge:** DevAssist — Finalize, evaluate, document, present.

---

## Core Principle

> **A system is not complete when it produces correct outputs on a few examples.**
> It is complete when it can be **evaluated, monitored, maintained, and deployed responsibly.**

## Lab Structure

```
Part 1 (40 min): EVALUATE  — run automated test suite + human evaluation
Part 2 (25 min): DOCUMENT  — final report, risk register, usage policy
Part 3 (25 min): PRESENT   — 5-min demo + 3-min Q&A per pair
```

---
## Setup — Rebuild the Full Pipeline

In [ ]:
import json, sys, os, time, re
sys.path.insert(0, '.')

from utils.generation_utils import generate, is_ollama_available
from utils.chunking_utils import load_and_chunk_corpus
from utils.retrieval_utils import format_context, query_collection
from utils.security_utils import (
    sanitize_input, validate_output,
    HARDENED_SYSTEM_PROMPT, ORIGINAL_SYSTEM_PROMPT
)
from evaluation.test_suite import run_test_suite, print_results

OLLAMA_OK = is_ollama_available()
SBERT_OK = CHROMA_OK = False
try:
    from sentence_transformers import SentenceTransformer
    SBERT_OK = True
except ImportError: pass
try:
    import chromadb
    CHROMA_OK = True
except ImportError: pass

with open('data/precomputed_outputs.json') as f:
    PRECOMPUTED = json.load(f)

print(f"Ollama: {'✓' if OLLAMA_OK else '⚠'}  |  SBERT: {'✓' if SBERT_OK else '⚠'}  |  Chroma: {'✓' if CHROMA_OK else '⚠'}")

In [ ]:
# Build index
embedding_model = None
collection = None

if SBERT_OK and CHROMA_OK:
    embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
    client = chromadb.Client()
    try: client.delete_collection('taskflow_docs')
    except: pass
    collection = client.create_collection('taskflow_docs', metadata={'hnsw:space': 'cosine'})
    all_chunks = load_and_chunk_corpus('corpus/docs', strategy='sections', min_length=80)
    chunk_texts = [c['text'] for c in all_chunks]
    chunk_ids = [f'chunk_{i}' for i in range(len(all_chunks))]
    chunk_metas = [{'source_file': c['source_file'], 'section': c.get('section','')} for c in all_chunks]
    chunk_embs = embedding_model.encode(chunk_texts).tolist()
    collection.add(ids=chunk_ids, documents=chunk_texts, embeddings=chunk_embs, metadatas=chunk_metas)
    print(f'✓ Indexed {collection.count()} chunks')
else:
    print('⚠ Using precomputed outputs')

LIVE = OLLAMA_OK and collection is not None

In [ ]:
# Full pipeline query function
def devassist_query(question, top_k=3):
    """Full DevAssist pipeline: sanitize → retrieve → generate → validate."""
    result = {'question': question, 'timestamp': time.strftime('%Y-%m-%d %H:%M:%S')}

    # Input sanitization
    san = sanitize_input(question)
    result['sanitization'] = san
    if san['blocked']:
        result['response'] = 'Query blocked by safety system. Please rephrase.'
        result['blocked'] = True
        return result

    # Retrieve
    passages = query_collection(collection, embedding_model, question, top_k=top_k)
    context_block = format_context(passages)
    prompt = HARDENED_SYSTEM_PROMPT.format(context=context_block, user_question=question)

    # Generate
    start = time.time()
    raw = generate(prompt, temperature=0.2)
    result['latency_ms'] = round((time.time() - start) * 1000)
    result['raw_response'] = raw
    result['retrieved_passages'] = passages

    # Validate
    val = validate_output(raw, passages)
    result['validation'] = val
    result['response'] = val['filtered_response']
    result['blocked'] = not val['passed']
    return result

# Quick test
if LIVE:
    t = devassist_query('How do I authenticate?')
    print(f'✓ Pipeline working. Latency: {t["latency_ms"]}ms')
    print(f'  Response: {t["response"][:200]}...')
else:
    print('✓ Using precomputed outputs for evaluation.')

---
# PART 1 — EVALUATE (40 min)

---
## 1.1 Run Automated Test Suite

In [ ]:
# Load test cases
with open('evaluation/test_cases.json') as f:
    test_cases = json.load(f)

print(f'Loaded {len(test_cases)} test cases:')
from collections import Counter
for cat, n in Counter(tc['category'] for tc in test_cases).items():
    print(f'  {cat}: {n}')

In [ ]:
# Generate responses for all test cases
responses = {}

for tc in test_cases:
    tc_id = tc['id']
    question = tc.get('question', '')

    if LIVE and question:
        result = devassist_query(question)
        responses[tc_id] = result['response']
        print(f'{tc_id}: {result["response"][:80]}...')
    else:
        responses[tc_id] = PRECOMPUTED['responses'].get(tc_id, '')
        print(f'{tc_id}: [precomputed] {responses[tc_id][:80]}...')

In [ ]:
# Run the automated test suite
results = run_test_suite(test_cases, responses)
print_results(results)

# Save results
with open('data/eval_results.json', 'w') as f:
    json.dump(results, f, indent=2)
print('\n✓ Results saved to data/eval_results.json')

### Automated Test Analysis

**Q1: What is the overall pass rate? Per category?**

TODO

**Q2: Which test failures are most concerning? Why?**

TODO

**Q3: What would you add to the test suite to improve coverage?**

TODO

---
## 1.2 Human Evaluation

Score 5 queries using the rubric in `evaluation/rubric.md`.

In [ ]:
# Select 5 representative queries for human evaluation
human_eval_queries = [
    'How do I authenticate with the TaskFlow API?',
    'What are the valid task state transitions?',
    'How do I run the test suite?',
    'What was fixed in version 2.3.0?',
    'How does TaskFlow handle concurrent updates?',
]

for i, q in enumerate(human_eval_queries, 1):
    if LIVE:
        r = devassist_query(q)
        resp = r['response']
    else:
        # Use precomputed or closest match
        key_map = {0: 'fmt_01', 1: 'fmt_02', 2: 'fmt_03', 3: 'fmt_04', 4: 'acc_01'}
        resp = PRECOMPUTED['responses'].get(key_map.get(i-1, 'fmt_01'), '[N/A]')

    print(f'\n{"="*60}')
    print(f'Query {i}: {q}')
    print(f'{"="*60}')
    print(f'{resp}')
    print(f'\nScore in evaluation/rubric.md:')
    print(f'  Factual: _/3  Grounded: _/3  Citation: _/3  Helpful: _/3  Concise: _/3')

### Human Evaluation Notes

**Which dimension scores highest across all 5 queries?**

TODO

**Which dimension scores lowest? What improvement would help most?**

TODO

**Why can't automated tests replace human evaluation for this system?**

TODO

---
## 1.3 Latency & Cost Estimate

In [ ]:
# Measure latency across queries
latency_queries = [
    'How do I authenticate?',
    'What are the task states?',
    'How do I install TaskFlow?',
    'What is the rate limit?',
    'What database is used in production?',
]

latencies = []
if LIVE:
    for q in latency_queries:
        r = devassist_query(q)
        latencies.append(r['latency_ms'])
        print(f'  {q[:40]:<42} {r["latency_ms"]:>6} ms')
else:
    latencies = list(PRECOMPUTED.get('latencies_ms', {}).values())[:5]
    for q, lat in zip(latency_queries, latencies):
        print(f'  {q[:40]:<42} {lat:>6} ms [precomputed]')

if latencies:
    import numpy as np
    print(f'\nLatency stats:')
    print(f'  Mean:   {np.mean(latencies):.0f} ms')
    print(f'  Median: {np.median(latencies):.0f} ms')
    print(f'  Std:    {np.std(latencies):.0f} ms')
    print(f'\nCost estimate for 100 queries/day:')
    avg_tokens = 200  # Approximate
    print(f'  ~{100 * avg_tokens:,} tokens/day generation')
    print(f'  ~{100:,} embedding operations/day')
    print(f'  Total latency: ~{100 * np.mean(latencies) / 1000:.0f} seconds/day')

---
# PART 2 — DOCUMENT (25 min)

Complete the following deliverables:

1. **`evaluation/evaluation.md`** — Fill in test results and known limitations
2. **`evaluation/rubric.md`** — Fill in scoring sheet from human evaluation
3. **`evaluation/results.md`** — Fill in automated + human results summary
4. **`docs/final_report.md`** — Complete all sections with mechanistic justifications
5. **`docs/risk_register.md`** — Update status and add any new risks discovered
6. **`docs/usage_policy.md`** — Verify and finalize
7. **`docs/portfolio_reflection.md`** — Complete capstone reflection

---
## Pre-Flight Checklist

Before presenting, verify all deliverables:

In [ ]:
# Pre-flight checklist
checks = [
    ('devassist.py', 'CLI entry point'),
    ('prompt_templates/system_prompt_v1.0.md', 'Prompt v1.0'),
    ('prompt_templates/system_prompt_v1.1.md', 'Prompt v1.1 (hardened)'),
    ('prompt_templates/CHANGELOG.md', 'Prompt changelog'),
    ('evaluation/evaluation.md', 'Evaluation strategy'),
    ('evaluation/rubric.md', 'Human evaluation rubric'),
    ('evaluation/results.md', 'Evaluation results'),
    ('evaluation/test_cases.json', 'Test cases'),
    ('evaluation/test_suite.py', 'Automated test suite'),
    ('docs/final_report.md', 'Final report'),
    ('docs/risk_register.md', 'Risk register'),
    ('docs/usage_policy.md', 'Usage policy'),
    ('docs/demo_script.md', 'Demo script'),
    ('docs/portfolio_reflection.md', 'Portfolio reflection'),
]

print('Pre-flight checklist:')
all_ok = True
for path, desc in checks:
    exists = os.path.exists(path)
    if exists:
        with open(path) as f:
            content = f.read()
        todos = content.count('TODO')
        status = f'✅ ({todos} TODOs)' if todos <= 3 else f'⚠️  {todos} TODOs remaining'
    else:
        status = '❌ MISSING'
        all_ok = False
    print(f'  {status:<20} {desc:<30} {path}')

if all_ok:
    print('\n🎉 All files present!')
else:
    print('\n⚠️  Some files missing — create them before presenting.')

---
# PART 3 — PRESENT (25 min)

## Format: 5-min demo + 3-min Q&A per pair

See `docs/demo_script.md` for the presentation template.

### Presentation Rubric

| Criterion | Excellent | Adequate | Insufficient |
|-----------|-----------|----------|-------------|
| Demo shows working pipeline | End-to-end with all 3 layers visible | Works with minor issues | Doesn't run |
| Mechanistic justification | Every choice traced to Module 1-7 | Most choices justified | No mechanistic reasoning |
| Evaluation presented | Test suite + human eval + limitations | Partial evaluation | No evaluation |
| Q&A defense | Fluent, correct, mechanistic | Mostly correct | Cannot defend choices |

---
# WRAP-UP

## Final Commit

```bash
git add -A && git commit -m 'Lab 8 — Ship It: final capstone delivery' && git push
```

## Course Takeaways

1. TODO
2. TODO
3. TODO

---

> *"Ship good systems. Know their limitations. Document what you don't know."*

---
*Lab 8 of 8 — DevAssist / TaskFlow Lab Series*